# Airline Reviews NLP Pipeline

Union two review datasets, clean and transform text, and run NLP analysis for later modeling.

Phases:
- Phase 1: collecte des Données
- Phase 2: preprocessing, tokenization, and vectorization
- Phase 3: sentiment, topics, and NER

In [1]:
# 1) Import Libraries and Set Paths
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Optional NLP libs
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.sentiment import SentimentIntensityAnalyzer
    nltk.download("stopwords", quiet=True)
    nltk.download("vader_lexicon", quiet=True)
    NLTK_AVAILABLE = True
except Exception:
    NLTK_AVAILABLE = False

try:
    import spacy
    SPACY_AVAILABLE = True
except Exception:
    SPACY_AVAILABLE = False

try:
    from transformers import pipeline, AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
except Exception:
    TRANSFORMERS_AVAILABLE = False

RAW_PATH = Path(r"C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\airline_reviews_raw_data.csv")
SCRAPED_PATH = Path(r"C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\airline_reviews_scraped.csv")
OUTPUT_DIR = Path(r"C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("NLTK:", NLTK_AVAILABLE, "SPACY:", SPACY_AVAILABLE, "TRANSFORMERS:", TRANSFORMERS_AVAILABLE)
print("Output:", OUTPUT_DIR)

NLTK: False SPACY: False TRANSFORMERS: False
Output: C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp


In [2]:
# 2) Load Raw and Scraped CSV Files
def safe_read_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")

df_raw = safe_read_csv(RAW_PATH)
df_scraped = safe_read_csv(SCRAPED_PATH)

print("Raw shape:", df_raw.shape)
print("Scraped shape:", df_scraped.shape)
display(df_raw.head(3))
display(df_scraped.head(3))
print("Raw columns:", df_raw.columns.tolist())
print("Scraped columns:", df_scraped.columns.tolist())

Raw shape: (5000, 6)
Scraped shape: (897, 6)


,airline,source,review_text,rating,date,verified_purchase
0,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was excellent. Food and Drink was...,2,2026-05-02,yes
1,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was good. Food and Drink was aver...,4,2026-05-02,yes
2,Talyena Airlines,repo_satisfaction_ratings,Seat Comfort was excellent. Food and Drink was...,4,2026-05-02,yes


,airline,source,review_text,rating,date,verified_purchase
0,Lufthansa,pullpush_reddit,Airlines Are Collecting Your Data And Selling ...,4,2026-05-02,unknown
1,Southwest Airlines,pullpush_reddit,Explore Multi-City Flight Packages with Car an...,5,2026-05-02,unknown
2,Qatar Airways,pullpush_reddit,"Critical News Committee - May 7th, 2025. https...",5,2026-05-02,unknown


Raw columns: ['airline', 'source', 'review_text', 'rating', 'date', 'verified_purchase']
Scraped columns: ['airline', 'source', 'review_text', 'rating', 'date', 'verified_purchase']


In [3]:
# 3) Union the Datasets and Save Combined CSV
all_cols = sorted(set(df_raw.columns).union(set(df_scraped.columns)))
df_raw_aligned = df_raw.reindex(columns=all_cols)
df_scraped_aligned = df_scraped.reindex(columns=all_cols)

df_all = pd.concat([df_raw_aligned, df_scraped_aligned], ignore_index=True)
before_dupes = len(df_all)
df_all = df_all.drop_duplicates()
after_dupes = len(df_all)

combined_path = OUTPUT_DIR / "airline_reviews_combined.csv"
df_all.to_csv(combined_path, index=False)

print(f"Combined rows: {before_dupes} -> {after_dupes}")
print(f"Saved: {combined_path}")

Combined rows: 5897 -> 4541
Saved: C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp\airline_reviews_combined.csv


In [4]:
# 4) Text Cleaning: Normalize, Remove Duplicates and Stopwords
def detect_text_column(columns):
    candidates = ["review", "review_text", "text", "content", "comment", "comments", "feedback", "body", "title"]
    cols_lower = {c.lower(): c for c in columns}
    for c in candidates:
        if c in cols_lower:
            return cols_lower[c]
    for c in columns:
        if any(k in c.lower() for k in candidates):
            return c
    return None

text_col = detect_text_column(df_all.columns)
if text_col is None:
    raise ValueError("Could not detect a review text column.")
print("Text column:", text_col)

stop_words = set()
if NLTK_AVAILABLE:
    stop_words = set(stopwords.words("english"))
else:
    stop_words = {"the","a","an","is","are","was","were","to","and","or","in","on","for","of","with"}

def clean_text(s):
    s = "" if pd.isna(s) else str(s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_all["text_clean"] = df_all[text_col].apply(clean_text)
df_all = df_all.drop_duplicates(subset=["text_clean"]).reset_index(drop=True)

def remove_stopwords(s):
    tokens = [t for t in s.split() if t not in stop_words]
    return " ".join(tokens)

df_all["text_nostop"] = df_all["text_clean"].apply(remove_stopwords)
print("Rows after cleaning:", len(df_all))

Text column: review_text
Rows after cleaning: 4440


In [5]:
# 5) Tokenization and Vectorization (TF-IDF and Embeddings)
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_all["text_nostop"])

tfidf_path = OUTPUT_DIR / "tfidf_matrix.npz"
vocab_path = OUTPUT_DIR / "tfidf_vocab.json"
try:
    from scipy import sparse
    sparse.save_npz(tfidf_path, tfidf_matrix)
    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(tfidf.vocabulary_, f)
    print(f"Saved TF-IDF: {tfidf_path}")
except Exception as e:
    print(f"TF-IDF save skipped: {e}")

USE_EMBEDDINGS = False
embeddings_path = OUTPUT_DIR / "text_embeddings.npy"
if USE_EMBEDDINGS and TRANSFORMERS_AVAILABLE:
    import torch
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained("distilbert-base-uncased")

    def mean_pool(last_hidden, attn_mask):
        mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        masked = last_hidden * mask
        return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    texts = df_all["text_clean"].tolist()
    batch_size = 32
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        tok = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**tok)
        embeds = mean_pool(outputs.last_hidden_state, tok["attention_mask"]).cpu().numpy()
        all_embeds.append(embeds)
    all_embeds = np.vstack(all_embeds)
    np.save(embeddings_path, all_embeds)
    print(f"Saved embeddings: {embeddings_path}")
elif USE_EMBEDDINGS:
    print("Embeddings skipped: transformers not available.")

TF-IDF save skipped: Object of type int64 is not JSON serializable


In [6]:
# 6) Sentiment Analysis (VADER and DistilBERT)
df_all["sent_vader"] = np.nan
if NLTK_AVAILABLE:
    sia = SentimentIntensityAnalyzer()
    df_all["sent_vader"] = df_all["text_clean"].apply(lambda s: sia.polarity_scores(s)["compound"])
else:
    print("VADER skipped: NLTK not available.")

df_all["sent_bert_label"] = None
df_all["sent_bert_score"] = np.nan
USE_BERT = True
if USE_BERT and TRANSFORMERS_AVAILABLE:
    sent_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    sample_limit = min(2000, len(df_all))
    bert_out = sent_pipe(df_all["text_clean"].iloc[:sample_limit].tolist())
    df_all.loc[:sample_limit - 1, "sent_bert_label"] = [r["label"] for r in bert_out]
    df_all.loc[:sample_limit - 1, "sent_bert_score"] = [r["score"] for r in bert_out]
    print(f"DistilBERT scored {sample_limit} rows.")
elif USE_BERT:
    print("DistilBERT skipped: transformers not available.")

VADER skipped: NLTK not available.
DistilBERT skipped: transformers not available.


In [7]:
# 7) Topic Modeling with LDA
count_vec = CountVectorizer(max_features=3000, stop_words="english")
dtm = count_vec.fit_transform(df_all["text_nostop"])

n_topics = 8
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
topic_matrix = lda.fit_transform(dtm)
df_all["topic_id"] = topic_matrix.argmax(axis=1)

terms = np.array(count_vec.get_feature_names_out())
topics = []
for k, comp in enumerate(lda.components_):
    top_idx = comp.argsort()[::-1][:10]
    topics.append({"topic_id": k, "top_terms": ", ".join(terms[top_idx])})

topics_df = pd.DataFrame(topics)
topics_path = OUTPUT_DIR / "lda_topics.csv"
topics_df.to_csv(topics_path, index=False)
print(f"Saved topics: {topics_path}")

Saved topics: C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp\lda_topics.csv


In [8]:
# 8) Named Entity Recognition (NER)
df_all["entities"] = ""
sample_limit = min(1000, len(df_all))
texts_sample = df_all["text_clean"].iloc[:sample_limit].tolist()

if SPACY_AVAILABLE:
    try:
        nlp = spacy.load("en_core_web_sm")
    except Exception:
        nlp = None
        print("spaCy model not installed. Run: python -m spacy download en_core_web_sm")
    if nlp is not None:
        ents = []
        for doc in nlp.pipe(texts_sample, batch_size=50):
            ents.append(", ".join(sorted({f"{ent.text}:{ent.label_}" for ent in doc.ents})))
        df_all.loc[:sample_limit - 1, "entities"] = ents
elif TRANSFORMERS_AVAILABLE:
    ner_pipe = pipeline("ner", grouped_entities=True)
    ner_out = ner_pipe(texts_sample)
    ents = []
    for row in ner_out:
        ents.append(", ".join(sorted({f"{e['word']}:{e['entity_group']}" for e in row})))
    df_all.loc[:sample_limit - 1, "entities"] = ents
else:
    print("NER skipped: no spaCy or transformers available.")

NER skipped: no spaCy or transformers available.


In [9]:
# 9) Export Preprocessed Data and NLP Outputs
cleaned_path = OUTPUT_DIR / "airline_reviews_cleaned.csv"
df_all.to_csv(cleaned_path, index=False)

summary = {
    "rows": int(len(df_all)),
    "columns": df_all.columns.tolist(),
    "text_column": text_col,
    "tfidf_matrix_path": str(tfidf_path),
    "embeddings_path": str(embeddings_path) if USE_EMBEDDINGS else None,
    "topics_path": str(topics_path),
}
summary_path = OUTPUT_DIR / "nlp_export_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved cleaned data: {cleaned_path}")
print(f"Saved summary: {summary_path}")

Saved cleaned data: C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp\airline_reviews_cleaned.csv
Saved summary: C:\Users\achre\Downloads\Esprit\DL\SkyInsight\10_NLP\outputs_nlp\nlp_export_summary.json
